# Projet Challenge 2 – Découverte du jeu IBM TabFormer

Transactions de cartes bancaires synthétiques générées par IBM (environ 24 M de lignes, environ 2 000 utilisateurs,
plusieurs décennies). Objectif de ce notebook : **voir** les données et vérifier qu'elles couvrent tous les modules.

Exécute les cellules une par une (Maj + Entrée).

## 0. Téléchargement depuis Kaggle

In [ ]:
!pip -q install kagglehub
import kagglehub, os

chemin = kagglehub.dataset_download("ealtman2019/credit-card-transactions")
print("Dossier :", chemin)
for f in os.listdir(chemin):
    print(f"  {f:40s} {os.path.getsize(os.path.join(chemin, f)) / 1e6:8.1f} Mo")

Si Kaggle demande une authentification : sur Kaggle, **Settings → API → Create New Token**, puis dans Colab
**🔑 Secrets** (menu de gauche) ajoute `KAGGLE_USERNAME` et `KAGGLE_KEY`, exécute la cellule suivante et relance le téléchargement.

In [ ]:
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

## 1. Premier coup d'œil (100 000 lignes)

On lit d'abord un petit morceau : c'est instantané et suffisant pour voir les colonnes.
Le tableau est **interactif** : tu peux trier, filtrer et faire défiler.

In [ ]:
import pandas as pd
from google.colab import data_table
data_table.enable_dataframe_formatter()   # tableaux interactifs dans Colab

fichier = os.path.join(chemin, "credit_card_transactions-ibm_v2.csv")
if not os.path.exists(fichier):  # le nom peut varier selon la version
    fichier = os.path.join(chemin, [f for f in os.listdir(chemin) if f.endswith(".csv") and "transaction" in f.lower()][0])
print("Fichier transactions :", fichier)

apercu = pd.read_csv(fichier, nrows=100_000)
apercu.head(5000)   # le tableau interactif affiche jusqu'à 5 000 lignes confortablement

In [ ]:
data_table.disable_dataframe_formatter()  # retour à l'affichage classique pour la suite
print(apercu.dtypes)
apercu.describe(include="all").T

**À regarder :** `Amount` est probablement du texte (`$134.09`) à convertir ; la date est répartie sur
`Year`, `Month`, `Day`, `Time` ; la cible `Is Fraud?` vaut `Yes` / `No`.

## 2. Chargement complet (2 à 4 minutes)

On charge tout en réduisant la mémoire : types catégoriels pour le texte répétitif, entiers courts pour les dates.

In [ ]:
import numpy as np

types = {"User": "int16", "Card": "int8", "Year": "int16", "Month": "int8", "Day": "int8",
         "Use Chip": "category", "Merchant City": "category", "Merchant State": "category",
         "MCC": "int16", "Errors?": "category", "Is Fraud?": "category"}
df = pd.read_csv(fichier, dtype=types, engine="pyarrow")

df["Amount"] = df["Amount"].str.replace("$", "", regex=False).astype("float32")
df["fraude"] = (df["Is Fraud?"] == "Yes").astype("int8")
# pyarrow lit "Time" comme des objets heure (datetime.time) : on repasse par le texte "HH:MM:SS"
heure = df["Time"].astype(str)
heure = heure.where(heure.str.len() > 5, heure + ":00")   # "HH:MM" -> "HH:MM:SS" si besoin
df["date"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=df["Day"])) + pd.to_timedelta(heure)
del heure

print(f"{len(df):,} transactions x {df.shape[1]} colonnes")
print(f"Mémoire : {df.memory_usage(deep=True).sum() / 1e9:.2f} Go")
print(f"Utilisateurs : {df['User'].nunique():,} | cartes : {df.groupby(['User', 'Card']).ngroups:,}")
print(f"Période : {df['date'].min()} → {df['date'].max()}")
print(f"Fraudes : {df['fraude'].sum():,} ({df['fraude'].mean() * 100:.3f} %)")

## 3. Valeurs manquantes

In [ ]:
manq = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
manq[manq > 0]

`Errors?` vide = transaction sans erreur (pas une vraie valeur manquante). `Merchant State` et `Zip` sont vides
pour les achats en ligne et à l'étranger : c'est une information en soi.

## 4. Évolution dans le temps : la séparation chronologique est-elle possible ?

In [ ]:
import matplotlib.pyplot as plt

par_an = df.groupby(df["date"].dt.year).agg(transactions=("fraude", "size"), fraudes=("fraude", "sum"))
par_an["taux_%"] = par_an["fraudes"] / par_an["transactions"] * 100

fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
par_an["transactions"].plot(ax=ax[0], title="Transactions par an")
par_an["taux_%"].plot(ax=ax[1], title="Taux de fraude par an (%)", color="crimson")
plt.tight_layout(); plt.show()
par_an.tail(12)

**À regarder :** le jeu couvre plusieurs décennies, mais les premières années ont très peu de transactions.
On devra sans doute ne garder qu'une période récente (par exemple les 5 dernières années) pour le projet.

## 5. Montants par classe

In [ ]:
display(df.groupby("fraude")["Amount"].describe().round(2))
print(f"Montants négatifs (remboursements) : {(df['Amount'] < 0).sum():,}")

fig, ax = plt.subplots(figsize=(8, 3.5))
for c, lab in [(0, "normale"), (1, "fraude")]:
    df.loc[(df["fraude"] == c) & (df["Amount"].between(0, 500)), "Amount"] \
      .plot.hist(bins=100, density=True, alpha=0.5, ax=ax, label=lab)
ax.set_title("Distribution des montants (0–500 $)"); ax.legend(); plt.show()

## 6. Variables lisibles pour les codes motifs

In [ ]:
def taux_par(col, n=10):
    t = df.groupby(col, observed=True)["fraude"].agg(transactions="size", taux="mean")
    t["taux"] = (t["taux"] * 100).round(3)
    return t.sort_values("taux", ascending=False).head(n)

print("--- Mode de paiement ---");      display(taux_par("Use Chip"))
print("--- Erreurs ---");               display(taux_par("Errors?"))
print("--- Catégorie marchand (MCC) ---"); display(taux_par("MCC"))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 3.5))
df.groupby(df["date"].dt.hour)["fraude"].mean().mul(100).plot.bar(ax=ax[0], title="Taux de fraude par heure (%)")
df.groupby(df["date"].dt.dayofweek)["fraude"].mean().mul(100).plot.bar(ax=ax[1], title="Taux de fraude par jour (0 = lundi) (%)")
plt.tight_layout(); plt.show()

## 7. Comportement par carte : la vélocité est-elle exploitable ?

In [ ]:
par_carte = df.groupby(["User", "Card"]).agg(transactions=("fraude", "size"), fraudes=("fraude", "sum"))
print(f"Transactions par carte : médiane {par_carte['transactions'].median():.0f}, max {par_carte['transactions'].max():,}")
print(f"Cartes touchées par au moins une fraude : {(par_carte['fraudes'] > 0).sum():,} sur {len(par_carte):,}")

# Exemple : l'historique d'une carte fraudée autour de sa première fraude
u, c = par_carte[par_carte["fraudes"] > 0].index[0]
carte = df[(df["User"] == u) & (df["Card"] == c)].sort_values("date")
i = carte.index.get_loc(carte[carte["fraude"] == 1].index[0])
carte.iloc[max(i - 8, 0): i + 8][["date", "Amount", "Use Chip", "Merchant City", "MCC", "Errors?", "fraude"]]

**À regarder :** les fraudes arrivent-elles en rafale, dans une autre ville, en ligne, avec des erreurs ?
C'est exactement ce que les variables de vélocité et les codes motifs devront capter.

## Bilan à remplir

| Critère | OK ? | Remarque |
|---|---|---|
| Période retenue pour le projet | | |
| Fraudes suffisantes en validation et en test | | |
| Vélocité par carte | | |
| Coût en montants | | |
| Variables lisibles (MCC, erreurs, mode de paiement, ville) | | |
| Taille de l'échantillon d'utilisateurs | | |